<img src="../../docs/site/_static/shot-logo.png" alt="SHOT" width="240">

# Watching SHOT solve a problem

Two short demonstrations of what SHOT does while it solves a problem, both built with SHOTpy callbacks:

1. **The solution process on a two-variable problem.** SHOT solves a small convex MINLP with cuts that a Python
   callback generates, and one figure per iteration shows the feasible set, the points of the MIP relaxations and the
   cuts that remove them.
2. **Primal and dual bounds in real time.** SHOT solves a convex MINLP from MINLPLib while a figure of the primal
   bound, the dual bound and the gap between them is redrawn during the solve.

The notebook needs SHOTpy, NumPy and Matplotlib. The second part reads `flay03m.osil` from the test instances of the
SHOT repository, so start Jupyter from the repository or one of its build directories, for example:

```bash
PYTHONPATH=build jupyter lab examples/SHOTpy/SolutionProcessDemo.ipynb
```

In [ ]:
import math
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib import font_manager
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

import SHOTpy

# The first MIP solver in this list that the SHOTpy build includes is used. With HiGHS or Cbc, SHOT always uses the
# multi-tree strategy; the strategy is set to multi-tree below for all solvers, so the iterations are comparable
SOLVER_NAMES = {SHOTpy.MIPSolver.Highs: "HiGHS", SHOTpy.MIPSolver.Gurobi: "Gurobi",
                SHOTpy.MIPSolver.Cplex: "CPLEX", SHOTpy.MIPSolver.Cbc: "Cbc"}
MIP_SOLVER = next((s for s in SOLVER_NAMES if s in SHOTpy.getSupportedMIPSolvers()), None)

if MIP_SOLVER is None:
    raise RuntimeError("This SHOTpy build includes no MIP solver")

SETTINGS = {
    "Output.Console.LogLevel": 6,  # Off
    "Dual.MIP.Solver": int(MIP_SOLVER),
    "Dual.TreeStrategy": 0,  # Multi-tree
    "Dual.MIP.NumberOfThreads": 1,  # One thread makes the search easier to follow
}


def new_solver(settings=SETTINGS):
    solver = SHOTpy.Solver()

    for name, value in settings.items():
        solver.updateSetting(name, value)

    # Otherwise the log level is applied only after a problem file has been read
    solver.updateLogLevels()

    return solver


# The colours and fonts of the SHOT brand: SHOT red for SHOT's own series, blue for other data, and neutral ink
SHOT_RED, DATA_BLUE, INK, INK_BODY, INK_MUTED = "#E21F25", "#2F6DB5", "#18181B", "#4A4743", "#6B6762"
RULE, FIGURE_FRAME, WHITE = "#BDB8AE", "#DAD6CD", "#FFFFFF"

fonts = {font.name for font in font_manager.fontManager.ttflist}
SHOT_STYLE = {
    "font.family": [f for f in ("IBM Plex Sans", "Arial", "DejaVu Sans") if f in fonts][:1] or ["sans-serif"],
    "figure.facecolor": WHITE, "axes.facecolor": WHITE,
    "text.color": INK, "axes.labelcolor": INK_BODY, "xtick.color": INK_MUTED, "ytick.color": INK_MUTED,
    "axes.edgecolor": RULE, "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "legend.fontsize": "small", "legend.labelcolor": INK_MUTED,
    "lines.solid_capstyle": "round", "lines.dash_capstyle": "round",
}

print(f"SHOTpy with {SOLVER_NAMES[MIP_SOLVER]} as the MIP solver")

## 1. The solution process on a two-variable problem

The problem is `shot_ex_jogo` from `test/data/shot_ex_jogo.gms`, with one continuous and one integer variable:

$$
\min_{x_1,\,x_2} \; -x_1 - x_2
$$

subject to

$$
2x_1 - 3x_2 \le 2, \qquad x_1 \in [1,20],\; x_2 \in \mathbb{Z} \cap [1,20],
$$
$$
c_1:\quad 0.15(x_1-8)^2 + 0.1(x_2-6)^2 + \frac{0.025\,e^{x_1}}{x_2^2} \le 5,
$$
$$
c_2:\quad \frac{1}{x_1} + \frac{1}{x_2} - \sqrt{x_1}\,\sqrt{x_2} \le -4.
$$

SHOT's own cut generation is switched off (`Dual.CutStrategy = OnlyExternal`), and every cut comes from the
`HyperplaneSelection` callback below. For each solution point of the MIP relaxation, the callback linearizes the
most violated nonlinear constraint at the point and adds the linearization as a cut. Since the constraints are
convex, the cut removes the point but no feasible solution. The callback also records the points and cuts of each
iteration for the figures.

In [ ]:
def build_jogo(solver):
    problem = SHOTpy.Problem(solver)
    problem.name = "shot_ex_jogo"

    x1 = problem.addVariable("x1", SHOTpy.VariableType.Real, 1.0, 20.0)
    x2 = problem.addVariable("x2", SHOTpy.VariableType.Integer, 1.0, 20.0)

    problem.setObjective(-x1 - x2, SHOTpy.ObjectiveDirection.Minimize)
    problem.addConstraint(2 * x1 - 3 * x2 <= 2, "l")
    problem.addConstraint(0.15 * (x1 - 8.0)**2 + 0.1 * (x2 - 6.0)**2 + 0.025 * SHOTpy.exp(x1) / x2**2 <= 5, "c1")
    problem.addConstraint(1 / x1 + 1 / x2 - x1**0.5 * x2**0.5 <= -4, "c2")
    problem.finalize()

    return problem


solver = new_solver({**SETTINGS,
                     "Model.Convexity.AssumeConvex": True,
                     "Model.Reformulation.Constraint.PartitionQuadraticTerms": 2,  # Never
                     "Dual.CutStrategy": 2})  # Only the cuts of the HyperplaneSelection callback

# The same problem is given as the original and the reformulated problem, so the points of the callback are in the
# variables x1 and x2
problem = build_jogo(solver)
assert solver.setProblem(problem, problem)

# The points and cuts of the callback, by the iteration whose MIP relaxation gave the points
iteration_log = {}


def generate_hyperplanes(ctx):
    if not ctx.solutionPoints or ctx.iterationNumber == 0:
        return

    # The callback is called after the MIP relaxation of the previous iteration has been solved
    snapshot = iteration_log.setdefault(ctx.iterationNumber - 1, {"points": [], "cuts": []})
    constraints = ctx.reformulatedProblem.nonlinearConstraints

    for solution_point in ctx.solutionPoints:
        point = solution_point.point
        snapshot["points"].append(tuple(point[:2]))

        violation = solution_point.maxDeviation.value
        constraint = next((c for c in constraints if c.index == solution_point.maxDeviation.index), None)

        if constraint is None or violation <= 0.0:
            continue

        # The linearization g(x0) + grad g(x0) (x - x0) <= 0 of the violated constraint g(x) <= 0 at x0
        gradient = constraint.calculateGradient(point)

        if not gradient:
            continue

        rhs = sum(value * point[i] for i, value in gradient.items()) - violation

        hyperplane = SHOTpy.ExternalHyperplane()
        hyperplane.variableIndexes = list(gradient.keys())
        hyperplane.variableCoefficients = list(gradient.values())
        hyperplane.rhsValue = rhs
        hyperplane.isGlobal = True
        hyperplane.source = SHOTpy.HyperplaneSource.External
        hyperplane.description = f"jogo_iteration{ctx.iterationNumber}_{constraint.name}"
        ctx.addHyperplane(hyperplane)

        snapshot["cuts"].append({"constraint": constraint.name, "point": tuple(point[:2]),
                                 "gradient": (gradient.get(0, 0.0), gradient.get(1, 0.0)), "rhs": rhs})

        break  # One cut per call is enough for this problem


solver.registerCallback(SHOTpy.CallbackLocation.HyperplaneSelection, generate_hyperplanes)
assert solver.solveProblem()

solution = solver.getPrimalSolution()
number_of_iterations = solver.getSolutionStatistics().numberOfIterations
number_of_cuts = sum(len(snapshot["cuts"]) for snapshot in iteration_log.values())

print(f"Status: {solver.getModelReturnStatus()}")
print(f"Optimal solution x1 = {solution.point[0]:.6f}, x2 = {solution.point[1]:.0f}, "
      f"objective {solution.objValue:.6f}")
print(f"{number_of_iterations} iterations, {number_of_cuts} cuts from the callback")

# The known optimum of shot_ex_jogo
assert solver.getModelReturnStatus() == SHOTpy.ModelReturnStatus.OptimalGlobal
assert abs(solution.objValue + 20.903615) < 1e-3
assert number_of_cuts > 0

### One figure per iteration

The green region satisfies all three constraints when $x_2$ is allowed to be continuous. The thick green segments are
the parts of it where $x_2$ is an integer, so together they are the feasible set of the MINLP. Dashed blue curves are
contours of the objective $-x_1 - x_2$, which decreases up and to the right.

In each figure, the black circles are the solution points of that iteration's MIP relaxation, the orange stars are
the points where the constraints are linearized, and the red lines are the new cuts. Grey points and faint red lines
are from earlier iterations. The cuts of an iteration are added to the MIP relaxation of the next one, so each
relaxation point lies inside all the earlier cuts, and the points approach the optimum from outside the feasible set.
The last figure also marks the optimal solution, and the axes are the same in every figure.

In [ ]:
# The constraint functions on a grid of the variable bounds
grid = np.linspace(1.0, 20.0, 240)
X, Y = np.meshgrid(grid, grid)
linear = 2 * X - 3 * Y - 2
c1 = 0.15 * (X - 8)**2 + 0.1 * (Y - 6)**2 + 0.025 * np.exp(X) / Y**2 - 5
c2 = 1 / X + 1 / Y - np.sqrt(X * Y) + 4
relaxed_feasible = (linear <= 0) & (c1 <= 0) & (c2 <= 0)

# The feasible parts of each row with integer x2; x1 is continuous, so these are segments
row_x = np.linspace(1.0, 20.0, 1500)
integer_rows = []

for y in range(1, 21):
    feasible = ((2 * row_x - 3 * y <= 2)
                & (0.15 * (row_x - 8)**2 + 0.1 * (y - 6)**2 + 0.025 * np.exp(row_x) / y**2 <= 5)
                & (1 / row_x + 1 / y - np.sqrt(row_x * y) <= -4))

    if feasible.any():
        integer_rows.append((y, np.where(feasible, row_x, np.nan)))

legend_handles = [
    Patch(facecolor="#d7efe1", label="feasible, x2 continuous"),
    Line2D([], [], color="#16834e", linewidth=3, label="feasible, x2 integer"),
    Line2D([], [], color="#6294c5", linestyle="--", label="objective contours"),
    Line2D([], [], color=INK, label="linear constraint l"),
    Line2D([], [], color="#d78a27", label="constraint c1"),
    Line2D([], [], color="#8856a7", label="constraint c2"),
    Line2D([], [], marker="o", markerfacecolor="none", color=INK, linestyle="None", label="relaxation point"),
    Line2D([], [], marker="*", color="#ee7b19", linestyle="None", label="linearization point"),
    Line2D([], [], color=SHOT_RED, label="new cut"),
    Line2D([], [], color="#b98c8c", label="earlier cut"),
    Line2D([], [], marker="D", color="#159447", linestyle="None", label="optimal solution"),
]


def plot_iteration(iteration):
    figure, ax = plt.subplots(figsize=(8.5, 5.1), constrained_layout=True)

    ax.contourf(X, Y, relaxed_feasible.astype(int), levels=[-0.5, 0.5, 1.5], colors=["#f8f1ee", "#d7efe1"])
    contours = ax.contour(X, Y, -X - Y, levels=np.arange(-36, -3, 4), colors="#6294c5", linestyles="dashed",
                          linewidths=0.8)
    ax.clabel(contours, inline=True, fontsize=7, fmt="%g")
    ax.contour(X, Y, linear, levels=[0], colors=INK, linewidths=1.2)
    ax.contour(X, Y, c1, levels=[0], colors="#d78a27", linewidths=1.2)
    ax.contour(X, Y, c2, levels=[0], colors="#8856a7", linewidths=1.2)

    for y in range(1, 21):
        ax.axhline(y, color=INK_MUTED, linestyle=":", linewidth=0.35, alpha=0.35)

    for y, xs in integer_rows:
        ax.plot(xs, np.full_like(xs, y), color="#16834e", linewidth=3, zorder=3)

    for recorded in sorted(k for k in iteration_log if k <= iteration):
        snapshot = iteration_log[recorded]
        current = recorded == iteration

        for cut in snapshot["cuts"]:
            gx, gy = cut["gradient"]
            ax.contour(X, Y, gx * X + gy * Y - cut["rhs"], levels=[0], colors=[SHOT_RED if current else "#b98c8c"],
                       linewidths=1.8 if current else 0.7, alpha=0.95 if current else 0.4)
            ax.scatter(*cut["point"], marker="*", s=125 if current else 35, color="#ee7b19" if current else "#c7a58a",
                       zorder=5)

        for point in snapshot["points"]:
            ax.scatter(*point, facecolors="none", edgecolors=INK if current else "#777777", s=75 if current else 25,
                       linewidths=1.4 if current else 0.7, zorder=6)

    if iteration == number_of_iterations:
        ax.scatter(*solution.point[:2], marker="D", s=65, color="#159447", edgecolors=INK, linewidths=0.5, zorder=7)

    cuts_so_far = sum(len(v["cuts"]) for k, v in iteration_log.items() if k <= iteration)
    ax.set(xlim=(0.8, 20.4), ylim=(0.8, 20.4), xlabel="x1 (continuous)", ylabel="x2 (integer)",
           title=f"Iteration {iteration} of {number_of_iterations}: {cuts_so_far} cuts")
    ax.set_aspect("equal", adjustable="box")
    ax.legend(handles=legend_handles, loc="upper left", bbox_to_anchor=(1.02, 1))
    plt.show()
    plt.close(figure)


with plt.rc_context(SHOT_STYLE):
    for iteration in range(1, number_of_iterations + 1):
        plot_iteration(iteration)

## 2. Primal and dual bounds in real time

SHOT keeps a primal bound, the objective value of the best solution found, and a dual bound, which no solution can
be better than. The problem is solved when the gap between them is small enough. Here SHOT solves `fo7`, a convex
MINLP from MINLPLib with 114 variables, of which 42 are binary, and 211 constraints, while a figure of the bounds is
redrawn during the solve.

The optimal solution of `fo7` is known: MINLPLib gives one with the objective value 20.729825. The
`PrimalCandidateSearch` callback gives it to SHOT as a candidate solution in the first iteration. SHOT checks the
candidate, and when it is feasible, the primal bound is optimal from the start, and the rest of the solve proves that
no better solution exists by raising the dual bound to it.

With Gurobi, SHOT uses the single-tree strategy: Gurobi solves one MIP problem, SHOT adds its cuts as lazy
constraints in Gurobi's callbacks, and Gurobi's bound is SHOT's dual bound, so the dual bound changes thousands of
times. With the other MIP solvers, SHOT uses the multi-tree strategy, the dual bound changes once per iteration, after
each MIP problem has been solved, and SHOT may not close the gap within the time limit.

The bounds are read in two callbacks:

* `TerminationCheck` is called once per iteration and also from inside the MIP solver while it solves the MIP
  problem, so the figure moves while SHOT is waiting for the MIP solver.
* `NewPrimalSolution` is called whenever SHOT has stored a new primal solution.

Every callback context has the current bounds and gaps, and a sample is recorded when they have changed. Redrawing a
figure takes much longer than a callback, so the figure is redrawn at most twice a second, and once more when the
solve has finished. The lower panel shows the relative gap on a logarithmic scale, and the dotted line is the relative
gap at which SHOT terminates.

With Gurobi, the solve takes 10 to 20 seconds, depending on the computer and the build. It is limited to `TIME_LIMIT`
seconds, and the time axis grows with the solve.

The solution of `fo7` from MINLPLib ([fo7.p1.sol](https://www.minlplib.org/sol/fo7.p1.sol)), by variable name. The
variables that are not listed are zero.

In [ ]:
MINLPLIB_OBJECTIVE = 20.7298250697881

MINLPLIB_SOLUTION = {name: float(value) for name, value in (pair.split("=") for pair in """
b2=1.0 b4=1.0 b6=1.0 b7=1.0 b9=1.0 b11=1.0 b14=1.0 b16=1.0 b17=1.0 b19=1.0 b23=1.0 b25=1.0 x44=2.26923076923077
x46=99999999.9999699 x47=99999999.9997983 x48=99999999.9999699 x49=99999999.9999021 x50=99999999.9999516
x51=100000000.0 x52=99999999.9999699 x53=100000000.0 x54=99999999.9999882 x55=100000000.0 x56=3.13538461538462
x57=1.52119458617611 x58=99999999.9999973 x59=99999999.9999021 x60=99999999.999979 x61=100000000.0
x62=99999999.9999973 x63=100000000.0 x64=100000000.000016 x65=100000000.0 x66=-1e-15 x67=6.49750096116878
x68=99999999.9999817 x69=100000000.000202 x70=100000000.0 x71=100000000.000202 x72=100000000.000018
x73=100000000.000202 x74=2.75717948717949 x75=1.52369362500733 x76=100000000.0 x77=100000000.000098
x78=100000000.000018 x79=100000000.000098 x80=1.51282051282051 x82=100000000.000037 x83=100000000.0
x84=1.51282051282051 x85=0.0 x86=2.26923076923077 x87=2.26923076923077 x88=4.00153846153846 x89=4.00153846153846
x90=1.51282051282051 x91=1.51282051282051 x92=1.51282051282051 x93=8.54 x94=7.05084745762712 x95=7.05084745762712
x96=3.99846212995002 x97=8.99653979238754 x98=5.94915254237288 x99=5.94915254237288 x100=5.94915254237288 x101=13.0
x102=1.13461538461538 x103=3.40384615384615 x104=6.53923076923077 x105=6.53923076923077 x106=3.78205128205128
x107=2.26923076923077 x108=0.756410256410256 x109=9.47457627118644 x110=9.47457627118644 x111=10.9957708573626
x112=4.49826989619377 x113=2.97457627118644 x114=2.97457627118644 x115=2.97457627118644
""".split())}

In [ ]:
from IPython.display import display

TIME_LIMIT = 60.0
REFRESH_INTERVAL = 0.5

# The test instances of the repository, from the repository or one of its build directories
relative_path = Path("test/data/instances/MINLP-convex/fo7.osil")
problem_file = next((base / relative_path for base in (Path.cwd(), *Path.cwd().parents)
                     if (base / relative_path).is_file()), None)

if problem_file is None:
    raise FileNotFoundError(f"Start the notebook from the SHOT repository or a build directory to find {relative_path}")

if SHOTpy.MIPSolver.Gurobi in SHOTpy.getSupportedMIPSolvers():
    bound_settings = {**SETTINGS, "Dual.MIP.Solver": int(SHOTpy.MIPSolver.Gurobi),
                      "Dual.TreeStrategy": 1}  # Single-tree
    print("Gurobi, single-tree")
else:
    bound_settings = SETTINGS
    print(f"{SOLVER_NAMES[MIP_SOLVER]}, multi-tree")

solver = new_solver({**bound_settings, "Termination.TimeLimit": TIME_LIMIT})
assert solver.setProblem(str(problem_file))

# The MINLPLib solution in the order of the variables of the problem
known_point = [MINLPLIB_SOLUTION.get(variable.name, 0.0) for variable in solver.getOriginalProblem().allVariables]
known_point_added = []
gap_tolerance = solver.getDoubleSetting("Termination.ObjectiveGap.Relative")

# (elapsed time, primal bound, dual bound, relative gap), one entry each time a bound has changed
samples = []


def finite(values):
    return [v if math.isfinite(v) else math.nan for v in values]


with plt.rc_context(SHOT_STYLE):
    figure, (bounds_axis, gap_axis) = plt.subplots(2, 1, figsize=(9, 6), sharex=True, height_ratios=(2, 1),
                                                   constrained_layout=True)

# Outside Jupyter there is no figure to update, and the figure is only shown at the end
display_handle = display(figure, display_id=True)
last_refresh = time.monotonic()


def refresh():
    bounds_axis.clear()
    gap_axis.clear()

    if samples:
        times, primal, dual, gaps = zip(*samples)
        # Each bound holds until the next sample, and the last one until now
        times = (*times, solver_time())
        primal, dual, gaps = (*primal, primal[-1]), (*dual, dual[-1]), (*gaps, gaps[-1])

        bounds_axis.step(times, finite(primal), where="post", color=SHOT_RED, linewidth=2, label="primal bound")
        bounds_axis.step(times, finite(dual), where="post", color=DATA_BLUE, linewidth=2, label="dual bound")
        bounds_axis.legend(loc="lower right")

        positive_gaps = [g if math.isfinite(g) and g > 0 else math.nan for g in gaps]

        if any(math.isfinite(g) for g in positive_gaps):
            gap_axis.step(times, positive_gaps, where="post", color=INK_BODY, linewidth=1.5)
            gap_axis.set_yscale("log")
            gap_axis.axhline(gap_tolerance, color=INK_MUTED, linestyle=":")

        primal_text = f"{primal[-1]:.4f}" if math.isfinite(primal[-1]) else "none"
        gap_text = f"{100 * gaps[-1]:.3g} %" if math.isfinite(gaps[-1]) else "none"
        bounds_axis.set_title(f"{problem_file.stem}: primal bound {primal_text}, dual bound {dual[-1]:.4f}, "
                              f"relative gap {gap_text}", loc="left")

    bounds_axis.set_ylabel("objective")
    gap_axis.set_ylabel("relative gap")
    gap_axis.set_xlabel("time (s)")
    gap_axis.set_xlim(0, max(1.0, 1.1 * solver_time()))

    if display_handle is not None:
        display_handle.update(figure)


solve_start = None


def solver_time():
    return time.monotonic() - solve_start


def record_bounds(ctx):
    global last_refresh

    sample = (solver_time(), ctx.primalBound, ctx.dualBound, ctx.relativeGap)

    if not samples or sample[1:] != samples[-1][1:]:
        samples.append(sample)

    if time.monotonic() - last_refresh >= REFRESH_INTERVAL:
        refresh()
        last_refresh = time.monotonic()


def add_known_solution(ctx):
    if not known_point_added:
        ctx.addPrimalSolution(known_point)
        known_point_added.append(ctx.iterationNumber)


solver.registerCallback(SHOTpy.CallbackLocation.PrimalCandidateSearch, add_known_solution)
solver.registerCallback(SHOTpy.CallbackLocation.TerminationCheck, record_bounds)
solver.registerCallback(SHOTpy.CallbackLocation.NewPrimalSolution, record_bounds)

solve_start = time.monotonic()

try:
    assert solver.solveProblem()
finally:
    # The bounds SHOT ends with can change after the last callback
    samples.append((solver_time(), solver.getPrimalBound(), solver.getGlobalDualBound(),
                    solver.getRelativeObjectiveGap()))
    refresh()

plt.close(figure)

if display_handle is None:
    display(figure)

_, primal_bound, dual_bound, relative_gap = samples[-1]
print(f"Status: {solver.getModelReturnStatus()} after {solver_time():.1f} s and "
      f"{solver.getSolutionStatistics().numberOfIterations} iterations, {len(samples)} bound changes recorded")
print(f"Primal bound {primal_bound:.6f}, dual bound {dual_bound:.6f}, relative gap {100 * relative_gap:.3g} %")

print(f"MINLPLib solution added in iteration {known_point_added[0]}, objective value {MINLPLIB_OBJECTIVE:.6f}")

# The MINLPLib solution is feasible, so SHOT's solution is at least as good, and the dual bound is valid
assert known_point_added and primal_bound <= MINLPLIB_OBJECTIVE + 1e-6
assert dual_bound <= primal_bound + 1e-6